# Soluciones · 03 Calibración SOL

Soluciones resueltas de los ejercicios del [cuaderno 03](../../notebooks/03_calibracion_sol.ipynb). Intenta antes cada ejercicio por tu cuenta.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting
from vnacourse.calibration import correct, solve_error_terms
from vnacourse.data import load_repetitions
from vnacourse.errormodel import ErrorTerms, apply_error_model, synthetic_error_terms
from vnacourse.reflection import gamma_to_z, z_to_gamma

plotting.use_course_style()

f_real, short_reps = load_repetitions("standards", "short")
_, open_reps = load_repetitions("standards", "open")
_, load_reps = load_repetitions("standards", "load")
f_real_mhz = f_real / 1e6
standards = (short_reps, open_reps, load_reps)
ideal = [-1.0, 1.0, 0.0]

f_hz = np.linspace(50e3, 1.45e9, 401)
ones = np.ones(f_hz.shape)
true_terms = synthetic_error_terms(f_hz)

## 1. Forma cerrada

Llama $s$, $o$ y $l$ a las lecturas en bruto del cortocircuito, el abierto y la carga, y escribe $t = e_{10}e_{01}$. El modelo $\Gamma_m = e_{00} + t\,\Gamma/(1 - e_{11}\Gamma)$ da:

- carga, $\Gamma = 0$: $\; l = e_{00}$
- abierto, $\Gamma = +1$: $\; (o - e_{00})(1 - e_{11}) = t$
- cortocircuito, $\Gamma = -1$: $\; (s - e_{00})(1 + e_{11}) = -t$

Sumar las dos últimas elimina $t$:

$$e_{11} = \frac{o + s - 2e_{00}}{o - s} \qquad\qquad t = (o - e_{00})(1 - e_{11})$$

In [ ]:
def solve_closed_form(short, open_, load):
    e00 = load
    e11 = (open_ + short - 2 * e00) / (open_ - short)
    tracking = (open_ - e00) * (1 - e11)
    return ErrorTerms(e00, e11, tracking)


measured = [reps.mean(axis=0) for reps in standards]
by_hand = solve_closed_form(*measured)
by_solver = solve_error_terms(measured, ideal)

for name, a, b in zip(ErrorTerms._fields, by_hand, by_solver, strict=True):
    print(f"{name:<13} mayor diferencia: {np.abs(a - b).max():.1e}")

Los mismos términos, hasta el error de redondeo, sin ninguna matriz. La forma cerrada muestra además para qué sirve cada estándar: la carga da directamente la directividad, y el cortocircuito y el abierto, al ser opuestos, separan la desadaptación de fuente (su suma) del seguimiento (su diferencia). Solo funciona con estándares *ideales*; la forma matricial acepta cualquier $\Gamma$ conocido.

## 2. Menos repeticiones

Calibra con la media de las repeticiones 1–5 y con la repetición 1 sola, y prueba ambas con la media de las repeticiones 6–10.

In [ ]:
test = [reps[5:].mean(axis=0) for reps in standards]
calibrations = {
    "cinco repeticiones": [reps[:5].mean(axis=0) for reps in standards],
    "una repetición": [reps[0] for reps in standards],
}

for label, train in calibrations.items():
    terms = solve_error_terms(train, ideal)
    print(label)
    for name, raw, target in zip(("cortocircuito", "abierto", "carga"), test, ideal, strict=True):
        residual = np.abs(correct(raw, terms) - target)
        print(f"   {name:<13} mediana {np.median(residual):.4f}   peor {residual.max():.3f}")

Con una sola repetición el residuo típico crece en un factor de aproximadamente 1.6, y el peor caso se duplica más o menos para el cortocircuito.

¿Por qué 1.6 y no $\sqrt{5} \approx 2.2$? El residuo contiene el ruido de la calibración *y* el ruido de los datos de prueba, que es una media de cinco en ambos casos. Si un barrido tiene ruido $\sigma$, los dos casos se comparan como $\sqrt{\sigma^2 + \sigma^2/5}$ frente a $\sqrt{\sigma^2/5 + \sigma^2/5}$, un cociente de $\sqrt{3} \approx 1.7$. Cerca de lo que vemos.

Promediar es barato y compensa, pero con rendimientos decrecientes: reducir el ruido a la mitad cuesta cuatro veces más barridos.

## 3. Dos parametrizaciones

In [ ]:
terms = solve_error_terms(measured, ideal)
ed, es = terms.directivity, terms.source_match
er = terms.tracking - ed * es

difference = np.abs(er - terms.tracking)
fig, ax = plt.subplots()
ax.semilogy(f_real_mhz, difference)
ax.set(xlabel="frecuencia (MHz)", ylabel="$|e_r - e_{10}e_{01}|$ (escala log.)")
ax.set_title("Diferencia entre las dos definiciones del tercer término")

worst = difference.argmax()
print(f"Mayor diferencia: {difference.max():.4f} a {f_real_mhz[worst]:.0f} MHz")
print(f"Mayor diferencia por encima de 150 MHz: {difference[f_real > 150e6].max():.1e}")

_, resistor = load_repetitions("dut", "load_75ohm")
raw = resistor.mean(axis=0)
other_form = (raw - ed) / (er + es * raw)
print("¿El mismo Γ corregido con ambas formas?", np.allclose(other_form, correct(raw, terms)))

Las dos magnitudes difieren exactamente en $e_{00}e_{11}$, el producto de la directividad y la desadaptación de fuente. Ese producto es máximo (unos 0.002) cerca de 64 MHz, en la banda donde ambos términos son grandes, y despreciable por encima de 150 MHz, donde ambos son pequeños.

El $\Gamma$ corregido es idéntico, porque ambas expresiones son la misma fórmula escrita con incógnitas distintas:

$$\frac{\Gamma_m - e_d}{e_r + e_s\Gamma_m} = \frac{\Gamma_m - e_{00}}{e_{10}e_{01} - e_{00}e_{11} + e_{11}\Gamma_m} = \frac{\Gamma_m - e_{00}}{e_{10}e_{01} + e_{11}(\Gamma_m - e_{00})}$$

Así que la elección no afecta a ningún resultado. Solo importa al *interpretar* o comparar los términos: $e_r$ no es, en sentido estricto, el seguimiento de reflexión. Para este instrumento la diferencia es como mucho del 0.2 %.

## 4. Un cuarto estándar

Con cuatro estándares el sistema tiene cuatro ecuaciones y tres incógnitas, y `np.linalg.lstsq` encuentra los términos que mejor se ajustan a las cuatro. Para ver si eso ayuda necesitamos ruido: sin él ambas soluciones son exactas. Añadimos un ruido de desviación típica 0.003 (la repetibilidad medida en el cuaderno 05) a las «medidas» de los estándares, calibramos, corregimos un dispositivo de 33 Ω sin ruido y repetimos muchas veces.

In [ ]:
def solve_least_squares(measured, actual):
    """Como solve_error_terms, para tres o más estándares."""
    actual = np.asarray(actual, dtype=complex)
    solution = np.empty((len(f_hz), 3), dtype=complex)
    for k in range(len(f_hz)):
        m = np.array([standard[k] for standard in measured])
        matrix = np.column_stack([np.ones(len(m)), actual * m, -actual])
        solution[k], *_ = np.linalg.lstsq(matrix, m, rcond=None)
    e00, e11, delta = solution.T
    return ErrorTerms(e00, e11, e00 * e11 - delta)


rng = np.random.default_rng(0)
noise_level = 0.003
known = [-1.0, 1.0, 0.0, float(z_to_gamma(75).real)]  # cortocircuito, abierto, carga, 75 Ω
device = z_to_gamma(33) * ones
device_raw = apply_error_model(device, true_terms)

errors = {3: [], 4: []}
for _ in range(200):
    noisy = [
        apply_error_model(gamma * ones, true_terms)
        + noise_level
        * (rng.standard_normal(f_hz.shape) + 1j * rng.standard_normal(f_hz.shape))
        / np.sqrt(2)
        for gamma in known
    ]
    for n in errors:
        terms_n = solve_least_squares(noisy[:n], known[:n])
        errors[n].append(np.abs(correct(device_raw, terms_n) - device))

for n, values in errors.items():
    print(f"{n} estándares: error RMS del Γ corregido = {np.sqrt(np.mean(np.square(values))):.4f}")

El cuarto estándar reduce en torno a una cuarta parte el error del dispositivo corregido. La ecuación adicional permite que el ruido de cada estándar se promedie en parte, y un estándar *dentro* de la carta restringe la calibración donde el cortocircuito y el abierto, ambos en el borde, no lo hacen.

Dos observaciones. La ganancia depende de dónde esté el dispositivo: es máxima para dispositivos en el centro de la carta y pequeña para los cercanos al borde (prueba con un dispositivo con $\Gamma = 0.8$). Y en la práctica el mayor beneficio de un cuarto estándar no es el ruido: con tres estándares el ajuste siempre es perfecto, así que no aprendes nada sobre su calidad; con cuatro, el residuo del ajuste te dice si tus estándares son coherentes entre sí.

## 5. ¿Cuánto importa el abierto?

Repite la sección 6 para tres valores de la capacidad de borde. Para ver el efecto del abierto por sí solo, lo ejecutamos también con un cortocircuito y una carga perfectos.

In [ ]:
omega = 2 * np.pi * f_hz
device_raw = apply_error_model(z_to_gamma(75) * ones, true_terms)

print(
    f"{'C del abierto':>14} {'Z a 1.45 GHz, solo el abierto imperfecto':>42} "
    f"{'como en la sección 6':>24}"
)
for capacitance in (0.05e-12, 0.1e-12, 0.2e-12):
    real_open = z_to_gamma(1 / (1j * omega * capacitance))
    row = []
    for real_short, real_load in (
        (-ones, 0 * ones),
        (z_to_gamma(1j * omega * 0.3e-9), z_to_gamma(51.5) * ones),
    ):
        seen = [apply_error_model(g, true_terms) for g in (real_short, real_open, real_load)]
        terms_ideal = solve_error_terms(seen, ideal)
        row.append(gamma_to_z(correct(device_raw, terms_ideal))[-1])
    print(f"{capacitance * 1e12:11.2f} pF {row[0]:>34.1f} Ω {row[1]:>22.1f} Ω")

Con solo el abierto imperfecto, la resistencia de 75 Ω adquiere una reactancia falsa de unos 0.9 Ω, 1.7 Ω y 3.4 Ω a 1.45 GHz: **proporcional a la capacidad** que hemos ignorado, mientras que la resistencia apenas se mueve. Una capacidad no modelada en el abierto aparece como un error de reactancia en todo lo que se mide después.

Si además el cortocircuito y la carga son los imperfectos de la sección 6, los errores se suman: la resistencia queda unos 2.3 Ω por debajo (la carga) y el error de reactancia crece aún más (la inductancia del cortocircuito empuja en el mismo sentido).

Para hacerse una idea: capacidades de este orden, de decenas a un par de cientos de femtofaradios, son las que tiene un conector coaxial abierto. A 1.45 GHz eso ya es un error del 1 % al 5 % en un dispositivo de 75 Ω, y crece proporcionalmente a la frecuencia. Por eso los kits de calibración se entregan con una definición de cada estándar.